In [1]:
!pip install hnswlib

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679799 sha256=2e5f752d5466c7f7f0aac0567091a571fb4194c5c2191af9aaeb1e481fb082df
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50e3ef0b67e
Successfully built hnswlib


In [2]:
	import hnswlib
	import numpy as np
	import time
	from sklearn.neighbors import NearestNeighbors

In [3]:
	# Build Dummy Dataset
	num_elements = 1000
	dim = 2
	data = np.random.random((num_elements, dim)).astype(np.float32)

	# Query point
	query = np.array([[0.5, 0.5]], dtype=np.float32)
	k = 5  # search for 5 nearest neighbors

In [4]:
# Build NN Model
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(data)

# Compute time performance
start = time.time()
distances, indices = nn.kneighbors(query)
end = time.time()

print("=== Exact NN ===")
print("Indices:", indices)
print("Distances:", distances)
print("Time:", end - start, "s")

=== Exact NN ===
Indices: [[932 849 282 236 858]]
Distances: [[0.01660996 0.0349302  0.04314451 0.04377912 0.04807957]]
Time: 0.06361198425292969 s


In [5]:
	# Initiate Index
	p = hnswlib.Index(space='l2', dim=dim)

	# Define Max Elements
	p.init_index(max_elements=num_elements, ef_construction=100, M=16)

	# Add Data
	p.add_items(data)

	# Set searching parameter
	p.set_ef(50)   # tradeoff speed vs accuracy

	start = time.time()
	labels, distances = p.knn_query(query, k=k)
	end = time.time()

	print("\n=== HNSW ===")
	print("Indices:", labels)
	print("Distances:", distances)
	print("Time:", end - start, "s")


=== HNSW ===
Indices: [[932 849 282 236 858]]
Distances: [[0.00027589 0.00122012 0.00186145 0.00191661 0.00231164]]
Time: 0.00021076202392578125 s
